# Carga de librerías

In [1]:
from dwca.read import DwCAReader
import pandas as pd

# Importando datos de plantas

Recogemos el archivo "core" y el archivo de multimedia del archivo DwCA.

In [2]:
with DwCAReader('E:/proyecto/data/raw/darwin-core-plants.zip') as dwca:
   references_df = dwca.pd_read('occurrence.txt', parse_dates=True) # Guarda información general de las imágenes.
   media_df = dwca.pd_read("multimedia.txt", parse_dates=True) # Guarda datos de las imágenes que queremos recopilar.

E:\proyecto\.venv\Lib\site-packages\dwca\read.py:255: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_or_textreader[shorten_term(field["term"])] = field_default_value


# Exploración de datos

Miramos los datos de "occurrences.txt".

In [3]:
first_entry = references_df.iloc[0]

for col in references_df.columns:
    if pd.notna(first_entry[col]):
        print(col, first_entry[col])

id 5110553501
license CC_BY_4_0
references https://identify.plantnet.org/k-macaronesia/observations/1026298856
basisOfRecord HUMAN_OBSERVATION
occurrenceID o-1026298856
individualCount 1
occurrenceStatus PRESENT
eventDate 2025-04-10T17:34:04Z
startDayOfYear 100
endDayOfYear 100
year 2025
month 4
day 10
continent AFRICA
countryCode ES
decimalLatitude 28.764942
decimalLongitude -17.77128
coordinateUncertaintyInMeters 5.0
scientificName Crambe strigosa L'Hér.
kingdom Plantae
phylum Tracheophyta
class Magnoliopsida
order Brassicales
family Brassicaceae
genus Crambe
genericName Crambe
specificEpithet strigosa
taxonRank SPECIES
taxonomicStatus ACCEPTED
datasetKey 7a3679ef-5582-4aaa-81f0-8c2545cafc81
publishingCountry FR
lastInterpreted 2025-11-25T15:38:30.607Z
elevation 413.7
elevationAccuracy 0.0
issue COORDINATE_ROUNDED;COUNTRY_DERIVED_FROM_COORDINATES;CONTINENT_DERIVED_FROM_COORDINATES
mediaType StillImage
hasCoordinate True
hasGeospatialIssues False
taxonKey 5373547
acceptedTaxonKey 5373

Y miramos el contenido del archivo "multimedia.txt".

In [4]:
first_entry = media_df.iloc[0]

for col in media_df.columns:
    if pd.notna(first_entry[col]):
        print(col, first_entry[col])

coreid 5110553501
type StillImage
format image/jpeg
identifier https://bs.plantnet.org/image/o/29068b4ed1e27583f0731f0723fa528dbffc6de0
title Crambe strigosa L'Hér.: leaf
description Crambe strigosa L'Hér.: leaf
source https://identify.plantnet.org/k-macaronesia/observations/1026298856
created 2025-04-10T17:34:04Z
creator Juan Carlos
license Juan Carlos (cc-by-sa)
rightsHolder Juan Carlos


# Creación de dataset

Cambiamos los nombre de algunas columnas y nos quedamos solo con los datos que nos interesan.

In [5]:
media_df.rename(columns={"coreid": "id", "identifier": "imageUrl"}, inplace=True)

In [6]:
columns = [
    "id", "species", "imageUrl", "description" # Nos quedamos con la especie (Género + especie) en lugar del nombre científico para facilitar la utilización de los datos posteriormente.
]

for col in references_df.columns:
    if col not in columns:
        references_df.drop(col, axis=1, inplace=True)

for col in media_df.columns:
    if col not in columns:
        media_df.drop(col, axis=1, inplace=True)

Y unimos ambos DataFrames.

In [7]:
merged_df = pd.merge(media_df, references_df, how="inner", on=["id"])

In [8]:
display(merged_df.head())

,id,imageUrl,description,species
0,5110553501,https://bs.plantnet.org/image/o/29068b4ed1e275...,Crambe strigosa L'Hér.: leaf,Crambe strigosa
1,5110553498,https://bs.plantnet.org/image/o/10d7f3f1e0b382...,Ageratina adenophora (Spreng.) R.M.King & H.Ro...,Ageratina adenophora
2,5110553498,https://bs.plantnet.org/image/o/5ec7534ab44618...,Ageratina adenophora (Spreng.) R.M.King & H.Ro...,Ageratina adenophora
3,5110553498,https://bs.plantnet.org/image/o/87fdc15ad23397...,Ageratina adenophora (Spreng.) R.M.King & H.Ro...,Ageratina adenophora
4,5110553498,https://bs.plantnet.org/image/o/8818f594a11070...,Ageratina adenophora (Spreng.) R.M.King & H.Ro...,Ageratina adenophora


Y finalmente, podemos contar el número de instancias de cada especie.

In [10]:
pd.set_option("display.max_rows", None)
print(merged_df.groupby("species").count().sort_values(by="id", ascending=False))
pd.set_option("display.max_rows", 10)

                                  id  imageUrl  description
species                                                    
Kleinia neriifolia               337       337          337
Rumex lunaria                    240       240          240
Euphorbia balsamifera            229       229          229
Solandra maxima                  192       192          192
Launaea arborescens              180       180          180
Lavandula canariensis            167       167          167
Astydamia latifolia              142       142          142
Canarina canariensis             139       139          139
Sonchus canariensis              125       125          125
Erica arborea                    122       122          122
Oxalis pes-caprae                117       117          117
Sonchus acaulis                  108       108          108
Periploca laevigata              100       100          100
Erysimum scoparium               100       100          100
Pinus canariensis                 98    

# Exportando dataset

In [11]:
merged_df.to_json("data/raw/flower_media.json")